In [204]:
from sklearn.preprocessing import MinMaxScaler
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM as lstm
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping
from keras import Input
import pmdarima as pm
from sklearn.metrics import mean_squared_error, mean_absolute_error
import pandas as pd
import numpy as np
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from statsmodels.tsa.seasonal import STL
from sklearn.linear_model import LinearRegression

# Modèles

## Persistence

In [173]:
def persistence(train, test=False): # pas d'ensemble de validation
    prediction = train.iloc[-1, 0]
    if type(test) != bool:
        # Évaluation du modèle
        n_test = len(test)
        predictions = prediction * np.ones(n_test)
        rmse = np.sqrt(mean_squared_error(test, predictions))
        mae = mean_absolute_error(test, predictions)
        return prediction, rmse, mae
    return prediction

## ARIMA

In [193]:
def ARIMA(train, saisonalite, test=False, type_produit=False): # pas d'ensemble de validation
    if type_produit:
        train = np.log(train)
    # fit
    fit_arima = pm.auto_arima(train, seasonal=saisonalite)

    if type_produit:
        train = np.exp(train) # on remet le train comme avant
    
    if type(test) != bool:
        # Évaluation du modèle
        n_test = len(test)
        predictions = fit_arima.predict(n_periods=n_test)

        # retour à l'échelle originale
        if type_produit:
            predictions = np.exp(predictions)
        
        rmse_lstm = np.sqrt(mean_squared_error(test, predictions))
        mae_lstm = mean_absolute_error(test, predictions)
        
        return fit_arima, rmse_lstm, mae_lstm
    return fit_arima

## STL + ARIMA sur le résidu

In [210]:
def STL_ARIMA(train, saisonalite, s_window, test=False, type_produit=False): # s_window : largeur de la fenetre saisonniere lors du lissage
    # local pour déterminer la saisonalité
    
    # Transformation log si multiplicatif
    if type_produit:
        train = np.log(train)
    
    # décomposition STL
    stl = STL(train, period=saisonalite, seasonal=s_window)
    
    if type_produit:
        train = np.exp(train) # on remet le train comme avant
    
    stl_result = stl.fit()
    
    trend = stl_result.trend
    seasonal = stl_result.seasonal
    resid = stl_result.resid

    # extrapolation de la tendance
    t = np.arange(len(trend)).reshape(-1,1)
    model_trend = LinearRegression().fit(t, trend)

    # saisonnalité
    season_pattern = seasonal.iloc[-saisonalite:].values
    
    # fit ARIMA sur les résidus
    fit_resid = pm.auto_arima(resid, seasonal=False)
    
    if type(test) != bool:
        n_test = len(test)

        # Prévision des résidus
        pred_resid = fit_resid.predict(n_periods=n_test)
    
        # tendance future
        t_future = np.arange(len(trend), len(trend)+n_test).reshape(-1,1)
        trend_future = model_trend.predict(t_future)
    
        # répétition de la saisonnalité 
        season_future = np.tile(season_pattern, int(np.ceil(n_test/saisonalite)))[:n_test] # tile répète le motif saisonnier
    
        # reconstruction
        y_pred = trend_future + season_future + pred_resid
    
        # Retour échelle originale
        if type_produit:
            y_pred = np.exp(y_pred)
    
        rmse = np.sqrt(mean_squared_error(test, y_pred))
        mae = mean_absolute_error(test, y_pred)
        
        return model_trend, season_pattern, fit_resid, rmse, mae
    
    return model_trend, season_pattern, fit_resid

## LSTM

In [175]:
def create_sequences(data, taille_fenêtre):
    
    X = []
    y = []

    # on parcourt la série
    for i in range(len(data) - taille_fenêtre):
        
        # séquence d'entrée
        X.append(data[i:i+taille_fenêtre])
        
        # valeur à prédire
        y.append(data[i+taille_fenêtre])

    return np.array(X), np.array(y)

In [176]:
def LSTM(train, val, saisonalite, test=False, n_neurones = 32, optimizer="adam", loss="mse", metrics=["mae"], epochs=80, batch_size=16, patience=10):
    # test = False si on ne veut pas tester le modèle sur un ensmble de test

    # Le scaler transforme les données pour qu'elles soient entre 0 et 1.
    
    scaler = MinMaxScaler(feature_range=(0, 1))
    
    # On apprend le minimum et le maximum à partir du train seulement.
    # Cela évite d'utiliser des informations du futur.
    
    train_scaled = scaler.fit_transform(train)
    val_scaled = scaler.transform(val)

    if type(test) != bool:
        # On applique exactement la même transformation au jeu de test.
        test_scaled = scaler.fit_transform(test)


    
    # Création des séquences

    L = saisonalite

    # création des séquences pour train
    X_train, y_train = create_sequences(train_scaled, L)
    X_train = X_train.reshape((X_train.shape[0], X_train.shape[1], 1))
    X_val, y_val = create_sequences(val_scaled, L)
    X_val = X_val.reshape((X_val.shape[0], X_val.shape[1], 1))

    if type(test) != bool:
        # création des séquences pour test
        X_test, y_test = create_sequences(test_scaled, L)
        X_test = X_test.reshape((X_test.shape[0], X_test.shape[1], 1))

    # Les données sont transformées en séquences afin de permettre au LSTM
    # d'apprendre les dépendances temporelles.
    # 
    # On apprend au modèle : “Si les 12 derniers mois ressemblent à ceci → le mois suivant devrait être cela.”
    
    # Chaque entrée du modèle correspond à une fenêtre temporelle
    # contenant les L observations précédentes.
    #
    # La sortie correspond à la valeur suivante de la série.
    #
    # Cette transformation permet de reformuler le problème de prévision
    # en un problème supervisé adapté aux réseaux de neurones.

    # Construction du modèle LSTM

    # Création d'un modèle séquentiel
    model = Sequential()
    
    # Couche LSTM
    # 32 neurones = taille de la mémoire du réseau
    model = Sequential([
    Input(shape=(X_train.shape[1], 1)),
    lstm(n_neurones),
    Dense(1)
    ])

    model.compile(
    optimizer=optimizer,
    loss=loss,
    metrics=metrics
    )

    early_stop = EarlyStopping(
        monitor="val_loss",
        patience=patience,
        restore_best_weights=True
    )

    model.fit(
    X_train,
    y_train,
    epochs=epochs,  # nombre d’itérations d’apprentissage
    batch_size=batch_size,    # nombre d'exemples traités à la fois
    validation_data=(X_val,y_val),
    callbacks=[early_stop],
    shuffle=False # série temporelle
    )

    if type(test) != bool:
        # Évaluation du modèle
        predictions = model.predict(X_test)
    
        # Remettre les valeurs à l’échelle originale; Comme on a normalisé les données, il faut annuler la normalisation.
        predictions = scaler.inverse_transform(predictions)
        y_test_real = scaler.inverse_transform(y_test.reshape(-1,1))
    
        rmse_lstm = np.sqrt(mean_squared_error(y_test_real, predictions))
        mae_lstm = mean_absolute_error(y_test_real, predictions)
        
        return model, rmse_lstm, mae_lstm, scaler
    return model, scaler
    

## LSTM multi-pas direct

In [177]:
def create_sequences_multi(data, window_size, horizon):

    X = []
    y = []

    for i in range(len(data) - window_size - horizon + 1):

        X.append(data[i:i+window_size])
        y.append(data[i+window_size:i+window_size+horizon])

    return np.array(X), np.array(y)

In [178]:
def LSTM_multi_pas_direct(train, val, saisonalite, test=False, taille_fenetre=12, n_pas = 12, n_neurones = 32, optimizer="adam", 
                   loss="mse", metrics=["mae"], n_epochs=80, batch_size=16, patience=10):
    scaler = MinMaxScaler(feature_range=(0, 1))
    
    # On apprend le minimum et le maximum à partir du train seulement.
    # Cela évite d'utiliser des informations du futur.
    
    train_scaled = scaler.fit_transform(train)
    val_scaled = scaler.transform(val) 

    if len(val_scaled) < taille_fenetre + n_pas:
        print('erreur, il faut un ensemble de validation plus grand !')
        return(None)

    # Création des séquences

    L = saisonalite
    X_train_multi, y_train_multi = create_sequences_multi(train_scaled, L, H)
    X_train_multi = X_train_multi.reshape((X_train_multi.shape[0], X_train_multi.shape[1], 1))
    X_val_multi, y_val_multi = create_sequences_multi(val_scaled, L, H)
    X_val_multi = X_val_multi.reshape((X_val_multi.shape[0], X_val_multi.shape[1], 1))

    if type(test) != bool:
        # On applique exactement la même transformation au jeu de test.
        test_scaled = scaler.transform(test)
        if len(test_scaled) < taille_fenetre + n_pas:
            print('erreur, il faut un ensemble de test plus grand !')
            return(None)
        
        X_test_multi, y_test_multi = create_sequences_multi(test_scaled, L, H)
        X_test_multi = X_test_multi.reshape((X_test_multi.shape[0], X_test_multi.shape[1], 1))

    # MODELE DIRECT MULTI-PAS

    model = Sequential([
    Input(shape=(taille_fenetre, 1)),
    lstm(n_neurones),
    Dense(n_pas) # multipas direct
    ])
    
    model.compile(
        optimizer=optimizer,
        loss=loss,
        metrics=metrics
    )

    early_stop = EarlyStopping(
        monitor="val_loss",
        patience=patience,
        restore_best_weights=True
    )

    
    model.fit(
        X_train_multi,
        y_train_multi,
        epochs=n_epochs,
        batch_size=batch_size,
        validation_data=(X_val_multi, y_val_multi),
        shuffle=False,
        callbacks=[early_stop]
    )

    if type(test) != bool:
        # Évaluation du modèle
        predictions = model.predict(X_test_multi)
    
        # Remettre les valeurs à l’échelle originale; Comme on a normalisé les données, il faut annuler la normalisation.
        predictions = scaler.inverse_transform(predictions.reshape(-1,1)).reshape(predictions.shape)
        y_test_real = scaler.inverse_transform(y_test_multi.reshape(-1,1)).reshape(y_test_multi.shape)
    
        rmse_lstm = np.sqrt(mean_squared_error(y_test_real.flatten(), predictions.flatten()))
        mae_lstm = mean_absolute_error(y_test_real.flatten(), predictions.flatten())
        
        return model, rmse_lstm, mae_lstm, scaler
    
    return model, scaler    

## ETS/Holt-Winters

In [192]:
def ETS(train, saisonalite, test=False, freq='MS', type_produit=False): # freq='MS' : données mensuelles au début du mois (Month Start)
    # pas d'ensemble de validation

    # on définit la fréquence
    train.index = pd.DatetimeIndex(train.index, freq=freq)

    # Transformation log si multiplicatif
    if type_produit:
        train = np.log(train)
    
    hw_model = ExponentialSmoothing(
        train,
        trend="add",
        seasonal="add",
        seasonal_periods=saisonalite
    ).fit()

    if type_produit:
        train = np.exp(train) # on remet le train comme avant

    if type(test) != bool:
        # Évaluation du modèle
        hw_pred = hw_model.forecast(len(test))

        # retour à l'échelle originale si log
        if type_produit:
            hw_pred = np.exp(hw_pred)

        rmse_hw = np.sqrt(mean_squared_error(test, hw_pred))
        mae_hw = mean_absolute_error(test, hw_pred)
        
        return hw_model, rmse_hw, mae_hw
    
    return hw_model

# Exemple d'utilisation

In [148]:
import pandas as pd

# charger les données
url = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/airline-passengers.csv"
df = pd.read_csv(url)

# Conversion de la colonne de dates
df["Month"] = pd.to_datetime(df["Month"])
df.set_index("Month", inplace=True)

# taille totale
n_total = len(df)

# proportions
train_frac = 0.6
val_frac = 0.2
test_frac = 0.2

# indices de split
train_end = int(n_total * train_frac)
val_end   = int(n_total * (train_frac + val_frac))

# création des ensembles
train = df.iloc[:train_end]
val   = df.iloc[train_end:val_end]
test  = df.iloc[val_end:]

train_val = pd.concat([df.iloc[:train_end], df.iloc[train_end:val_end]])

## Peristence

In [165]:
prediction, rmse, mae = persistence(train_val, test)

In [168]:
print(rmse, mae)
prediction * np.ones(10)

93.13394136662937 81.44827586206897


array([491., 491., 491., 491., 491., 491., 491., 491., 491., 491.])

## ARIMA

In [134]:
fit_arima, rmse_arima, mae_arima = ARIMA(train_val, 12, test=test, type_produit=True) # 12 mois par an

In [135]:
print(rmse_arima, mae_arima)
predictions = np.exp(fit_arima.predict(n_periods = len(test)))
predictions[:10]

91.08020294210444 80.9647405658993


1958-08-01    499.365473
1958-09-01    479.177118
1958-10-01    453.920047
1958-11-01    438.316155
1958-12-01    443.182558
1959-01-01    448.102990
1959-02-01    453.078051
1959-03-01    458.108348
1959-04-01    463.194494
1959-05-01    468.337108
Freq: MS, dtype: float64

## STL + ARIMA

In [218]:
model_trend, season_pattern, fit_resid, rmse_stl_arima, mae_stl_arima = STL_ARIMA(train_val, 12, 13, test=test, type_produit=True)

In [219]:
print(rmse_stl_arima, mae_stl_arima)

52.80580450671259 49.693353157507254


In [225]:
def forecast_stl_arima(model_trend, season_pattern, fit_resid, n_train, n_test, saisonalite, type_produit=False):

    # Prévision des résidus
    pred_resid = fit_resid.predict(n_periods=n_test)

    # tendance
    t_future = np.arange(n_train, n_train+n_test).reshape(-1,1)
    trend_future = model_trend.predict(t_future)

    # répétition de la saisonnalité
    season_future = np.tile(season_pattern, int(np.ceil(n_test/saisonalite)))[:n_test] # tile répète le motif saisonnier

    # reconstruction
    y_pred = trend_future + season_future + pred_resid

    # Retour échelle originale
    if type_produit:
        y_pred = np.exp(y_pred)

    return y_pred

In [226]:
predictions = forecast_stl_arima(model_trend, season_pattern, fit_resid, len(train_val), len(test), 12, type_produit=True)
predictions[:10]

1958-08-01    520.636004
1958-09-01    451.464775
1958-10-01    394.417279
1958-11-01    349.361820
1958-12-01    391.813456
1959-01-01    401.400792
1959-02-01    376.098851
1959-03-01    440.538246
1959-04-01    435.440966
1959-05-01    449.851118
Freq: MS, dtype: float64

## LSTM

### Simple

In [136]:
fit_LSTM, rmse_LSTM, mae_LSTM, scaler_simple = LSTM(train, val, 12, test=test) # 12 mois par an

Epoch 1/80
5/5 ━━━━━━━━━━━━━━━━━━━━ 3s 124ms/step - loss: 0.1660 - mae: 0.3652 - val_loss: 0.8061 - val_mae: 0.8730
Epoch 2/80
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step - loss: 0.1091 - mae: 0.2866 - val_loss: 0.5713 - val_mae: 0.7252
Epoch 3/80
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step - loss: 0.0665 - mae: 0.2132 - val_loss: 0.3785 - val_mae: 0.5759
Epoch 4/80
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step - loss: 0.0359 - mae: 0.1456 - val_loss: 0.2264 - val_mae: 0.4221
Epoch 5/80
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step - loss: 0.0183 - mae: 0.1004 - val_loss: 0.1215 - val_mae: 0.2773
Epoch 6/80
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 86ms/step - loss: 0.0150 - mae: 0.0983 - val_loss: 0.0708 - val_mae: 0.2025
Epoch 7/80
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step - loss: 0.0199 - mae: 0.1176 - val_loss: 0.0594 - val_mae: 0.1882
Epoch 8/80
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 60ms/step - loss: 0.0211 - mae: 0.1213 - val_loss: 0.0644 - val_mae: 0.1955
Epoch 9/80
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step - loss: 0.0184 - mae: 0.1112 -

In [137]:
print(rmse_LSTM, mae_LSTM)

80.04042605474342 67.43662396599265


### Multi-pas direct

In [138]:
fit_LSTM_direct, rmse_LSTM_direct, mae_LSTM_direct, scaler_multi = LSTM_multi_pas_direct(train, val, 12, test=test) 

Epoch 1/80
4/4 ━━━━━━━━━━━━━━━━━━━━ 5s 285ms/step - loss: 0.1937 - mae: 0.3949 - val_loss: 1.0096 - val_mae: 0.9717
Epoch 2/80
4/4 ━━━━━━━━━━━━━━━━━━━━ 1s 59ms/step - loss: 0.1743 - mae: 0.3712 - val_loss: 0.9250 - val_mae: 0.9257
Epoch 3/80
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 57ms/step - loss: 0.1554 - mae: 0.3469 - val_loss: 0.8384 - val_mae: 0.8753
Epoch 4/80
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step - loss: 0.1361 - mae: 0.3201 - val_loss: 0.7468 - val_mae: 0.8176
Epoch 5/80
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step - loss: 0.1160 - mae: 0.2895 - val_loss: 0.6477 - val_mae: 0.7490
Epoch 6/80
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step - loss: 0.0948 - mae: 0.2544 - val_loss: 0.5409 - val_mae: 0.6652
Epoch 7/80
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step - loss: 0.0732 - mae: 0.2157 - val_loss: 0.4299 - val_mae: 0.5786
Epoch 8/80
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 53ms/step - loss: 0.0533 - mae: 0.1792 - val_loss: 0.3255 - val_mae: 0.4966
Epoch 9/80
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step - loss: 0.0387 - mae: 0.1508 -

In [139]:
print(rmse_LSTM_direct, mae_LSTM_direct)

91.40665271621418 74.70031187269423


### Prévision multi-pas LSTM récursif

In [140]:
def forecast_lstm(model, val, scaler, saisonalite, n_steps):

    # scaler comme dans ton entraînement
    val_scaled = scaler.transform(val)

    # dernière fenêtre
    last_window = val_scaled[-saisonalite:]

    preds = []

    for _ in range(n_steps):

        X = last_window.reshape(1, saisonalite, 1)

        pred = model.predict(X, verbose=0)

        preds.append(pred[0,0])

        # mise à jour de la fenêtre
        last_window = np.vstack([last_window[1:], pred])

    preds = np.array(preds).reshape(-1,1)

    # retour à l'échelle originale
    preds = scaler.inverse_transform(preds)

    return preds

In [141]:
predictions = forecast_lstm(
    fit_LSTM,
    val,
    scaler_simple,
    saisonalite=12,
    n_steps=10 # prédit les 10 prochaines périodes
)
predictions

array([[370.32574],
       [370.09482],
       [370.62387],
       [371.77728],
       [373.33694],
       [373.7874 ],
       [373.94012],
       [374.46298],
       [373.89532],
       [373.65256]], dtype=float32)

### Prévision multi-pas LSTM direct

In [142]:
def forecast_lstm_direct(model, val, scaler, saisonalite):
    # scaler comme dans ton entraînement
    val_scaled = scaler.transform(val)

    # dernière fenêtre
    last_window = val_scaled[-saisonalite:]

    # Mise en forme pour LSTM
    X = last_window.reshape(1, saisonalite, 1)

    # Prédiction directe
    preds = model.predict(X)

    # Retour à l’échelle originale
    preds = scaler.inverse_transform(preds.reshape(-1,1)).reshape(preds.shape)

    return preds.flatten()

In [143]:
predictions = forecast_lstm_direct(
    fit_LSTM_direct,   # modèle direct multi-step
    val,
    scaler_multi,
    saisonalite=12
)
predictions

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 291ms/step


array([355.79498, 310.79932, 319.64377, 324.48483, 389.1881 , 386.6233 ,
       420.47742, 425.3807 , 364.0534 , 411.6325 , 383.12775, 405.4492 ],
      dtype=float32)

## ETS/Holt-Winter

In [189]:
hw_model, rmse_hw, mae_hw = ETS(train_val, 12, type_produit=True, test=test)

In [190]:
print(rmse_hw, mae_hw)
predictions = np.exp(hw_model.forecast(len(test))) # type produit
predictions[:10]

37.499715417070334 33.063743206790235


1958-08-01    490.310384
1958-09-01    437.443148
1958-10-01    383.137436
1958-11-01    336.333840
1958-12-01    383.843538
1959-01-01    391.662281
1959-02-01    390.561367
1959-03-01    451.182090
1959-04-01    436.755115
1959-05-01    437.190162
Freq: MS, dtype: float64